# R3 Microstructure EDA — HYDROGEL_PACK + VELVETFRUIT_EXTRACT

Source: `notebooks/02_microstructure_eda.py` over `data/round_3/prices_round_3_day_{0,1,2}.csv` (~30k ticks/product). Plots cached in `docs/round_3/research/plots/02_*.png`. Findings writeup: `docs/round_3/research/04_microstructure.md`.

Answers Q1-Q8 from the analysis brief: depth profile, wall_mid stability, spread regimes, autocorr robustness, mean reversion, fill simulation, inventory drift, adverse selection.

## Setup

Imports, plot config, and constants (path globs, product list, plots dir).

In [ ]:
%matplotlib inline
from __future__ import annotations

import glob
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PRICE_GLOB = "data/round_3/prices_round_3_day_*.csv"
TRADE_GLOB = "data/round_3/trades_round_3_day_*.csv"
PRODUCTS = ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]
PLOTS_DIR = "docs/round_3/research/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

## Load data

CSVs are semicolon-separated. Concatenate per-day price/trade files, restrict to our two products, and build a global tick index `t = day*1e6 + timestamp`.

In [ ]:
def load_prices() -> pd.DataFrame:
    df = pd.concat([pd.read_csv(f, sep=";") for f in sorted(glob.glob(PRICE_GLOB))],
                   ignore_index=True)
    df = df[df["product"].isin(PRODUCTS)].copy()
    df["t"] = df["day"] * 1_000_000 + df["timestamp"]
    df = df.sort_values(["product", "t"]).reset_index(drop=True)
    # bid_volumes are abs sizes; ask_volumes too in this dataset
    for c in df.columns:
        if c.startswith(("bid_volume", "ask_volume")):
            df[c] = df[c].fillna(0)
    return df


def load_trades() -> pd.DataFrame:
    parts = []
    for f in sorted(glob.glob(TRADE_GLOB)):
        d = pd.read_csv(f, sep=";")
        day = int(f.rsplit("_", 1)[-1].split(".")[0])
        d["day"] = day
        parts.append(d)
    df = pd.concat(parts, ignore_index=True)
    df = df[df["symbol"].isin(PRODUCTS)].copy()
    df["t"] = df["day"] * 1_000_000 + df["timestamp"]
    return df

## Book features

`wall_mid` is the mid of the deepest level on each side (largest size; falls back to top if only L1 exists). Also derive spread, top mid, and per-side level counts.

In [ ]:
def per_prod(df: pd.DataFrame, prod: str) -> pd.DataFrame:
    return df[df["product"] == prod].copy().reset_index(drop=True)


def add_book_features(p: pd.DataFrame) -> pd.DataFrame:
    p["best_bid"] = p["bid_price_1"]
    p["best_ask"] = p["ask_price_1"]
    p["top_mid"] = (p["best_bid"] + p["best_ask"]) / 2
    p["spread"] = p["best_ask"] - p["best_bid"]
    # wall_mid: deepest level (largest size) on each side; fall back to top
    bid_px = p[["bid_price_1", "bid_price_2", "bid_price_3"]].to_numpy()
    bid_sz = p[["bid_volume_1", "bid_volume_2", "bid_volume_3"]].to_numpy()
    ask_px = p[["ask_price_1", "ask_price_2", "ask_price_3"]].to_numpy()
    ask_sz = p[["ask_volume_1", "ask_volume_2", "ask_volume_3"]].to_numpy()
    # for missing levels px is NaN; mask their sizes to -inf so argmax picks real ones
    bid_sz_m = np.where(np.isnan(bid_px), -np.inf, bid_sz)
    ask_sz_m = np.where(np.isnan(ask_px), -np.inf, ask_sz)
    bid_idx = np.argmax(bid_sz_m, axis=1)
    ask_idx = np.argmax(ask_sz_m, axis=1)
    rows = np.arange(len(p))
    deep_bid = bid_px[rows, bid_idx]
    deep_ask = ask_px[rows, ask_idx]
    p["wall_bid"] = deep_bid
    p["wall_ask"] = deep_ask
    p["wall_mid"] = (deep_bid + deep_ask) / 2
    # depth count: how many levels populated per side
    p["bid_levels"] = (~np.isnan(bid_px)).sum(axis=1)
    p["ask_levels"] = (~np.isnan(ask_px)).sum(axis=1)
    p["total_bid_size"] = np.nansum(bid_sz, axis=1)
    p["total_ask_size"] = np.nansum(ask_sz, axis=1)
    return p

Load everything and build per-product books with features attached.

In [ ]:
print("Loading prices...")
prices = load_prices()
print("Loading trades...")
trades = load_trades()  # used for sanity / not heavy

books = {}
for prod in PRODUCTS:
    p = per_prod(prices, prod)
    p = add_book_features(p)
    books[prod] = p
    print(f"  {prod}: {len(p)} ticks across days {sorted(p['day'].unique())}")

print(f"  trades loaded: {len(trades)} rows")

## Q1 — Order book depth profile

How many levels are typically populated per side, and what are typical sizes at each level? Tells us whether "wall mid" is meaningful (it is, when L2 size > L1).

In [ ]:
def q1_depth(books: dict) -> None:
    print("\n### Q1 — Order book depth profile")
    for prod, p in books.items():
        print(f"\n[{prod}]")
        for side in ("bid", "ask"):
            lc = p[f"{side}_levels"].value_counts(normalize=True).sort_index()
            print(f"  {side} levels distribution: " +
                  ", ".join(f"L{int(k)}={v:.1%}" for k, v in lc.items()))
        for L in (1, 2, 3):
            bm = p[f"bid_volume_{L}"].replace(0, np.nan).median()
            am = p[f"ask_volume_{L}"].replace(0, np.nan).median()
            print(f"  median size  L{L}: bid={bm}, ask={am}")
        deep_frac = ((p["bid_levels"] == 3) & (p["ask_levels"] == 3)).mean()
        print(f"  fraction ticks with both sides 3-deep: {deep_frac:.1%}")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (prod, p) in zip(axes, books.items()):
        for L in (1, 2, 3):
            ax.hist(p[f"bid_volume_{L}"].replace(0, np.nan).dropna(), bins=40,
                    alpha=0.5, label=f"bid L{L}")
        ax.set_title(f"{prod}: bid size by level")
        ax.set_xlabel("size")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(f"{PLOTS_DIR}/02_q1_depth.png", dpi=110)
    plt.show()

q1_depth(books)

## Q2 — Wall mid vs top-of-book mid

Compare the two mid candidates: do they differ enough to matter? Look at level offset, change-magnitude std, and frequency of zero-change ticks.

In [ ]:
def q2_wall_mid(books: dict) -> None:
    print("\n### Q2 — Wall mid vs top-of-book mid")
    for prod, p in books.items():
        diff = p["wall_mid"] - p["top_mid"]
        d_top = p["top_mid"].diff()
        d_wall = p["wall_mid"].diff()
        print(f"\n[{prod}]")
        print(f"  wall_mid - top_mid: mean={diff.mean():.3f}, std={diff.std():.3f}, "
              f"|>0|={ (diff.abs() > 0).mean():.1%}")
        print(f"  std of d(top_mid) ={d_top.std():.3f}")
        print(f"  std of d(wall_mid)={d_wall.std():.3f}")
        print(f"  ratio std(d_wall)/std(d_top)={d_wall.std()/d_top.std():.3f}")
        print(f"  P(|d_wall|=0)={ (d_wall == 0).mean():.1%}, "
              f"P(|d_top|=0)={ (d_top == 0).mean():.1%}")
    # plot a 1500-tick slice
    fig, axes = plt.subplots(2, 1, figsize=(11, 6))
    for ax, (prod, p) in zip(axes, books.items()):
        s = p.iloc[5000:6500]
        ax.plot(s["t"].values, s["top_mid"].values, alpha=0.6, lw=0.8, label="top_mid")
        ax.plot(s["t"].values, s["wall_mid"].values, alpha=0.9, lw=0.8, label="wall_mid")
        ax.set_title(prod)
        ax.legend()
    fig.tight_layout()
    fig.savefig(f"{PLOTS_DIR}/02_q2_wallmid_vs_topmid.png", dpi=110)
    plt.show()

q2_wall_mid(books)

## Q3 — Spread regimes

Distribution of bid-ask spread, plus per-day stats and an intraday-bucket heatmap. Want to know if spread is stationary (it is) and whether half_edge can be set as a constant.

In [ ]:
def q3_spread(books: dict) -> None:
    print("\n### Q3 — Spread regimes")
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for col, (prod, p) in enumerate(books.items()):
        sp = p["spread"]
        med = sp.median()
        print(f"\n[{prod}] spread median={med}, mean={sp.mean():.2f}, "
              f"std={sp.std():.2f}, min={sp.min()}, max={sp.max()}")
        vc = sp.value_counts(normalize=True).sort_index()
        print("  top spread values:", ", ".join(f"{int(k)}:{v:.1%}" for k, v in vc.head(8).items()))
        # by day
        per_day = p.groupby("day")["spread"].agg(["median", "mean", "std"])
        print(per_day)
        # heatmap day x intraday-quartile
        p["bucket"] = pd.cut(p["timestamp"], bins=10, labels=range(10))
        heat = p.groupby(["day", "bucket"], observed=True)["spread"].mean().unstack()
        axes[0, col].imshow(heat.values, aspect="auto", cmap="viridis")
        axes[0, col].set_title(f"{prod}: mean spread, day×bucket")
        axes[0, col].set_xlabel("intraday bucket"); axes[0, col].set_ylabel("day")
        axes[1, col].hist(sp, bins=range(int(sp.min()), int(sp.max()) + 2), alpha=0.8)
        axes[1, col].set_title(f"{prod}: spread hist")
    fig.tight_layout()
    fig.savefig(f"{PLOTS_DIR}/02_q3_spread.png", dpi=110)
    plt.show()

q3_spread(books)

## Q4 — Autocorrelation robustness

Lag-1+ autocorr of wall_mid returns, broken out per day and per intraday quarter. Want to confirm any mean-reversion signal isn't a one-day artifact.

In [ ]:
def autocorr(x: np.ndarray, lag: int) -> float:
    x = x[~np.isnan(x)]
    if len(x) < lag + 5:
        return np.nan
    return float(np.corrcoef(x[:-lag], x[lag:])[0, 1])


def q4_autocorr(books: dict) -> None:
    print("\n### Q4 — Autocorrelation robustness")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, (prod, p) in zip(axes, books.items()):
        print(f"\n[{prod}]")
        rets = p["wall_mid"].diff().to_numpy()
        # per-day lag-1
        for d in sorted(p["day"].unique()):
            sub = p[p["day"] == d]["wall_mid"].diff().to_numpy()
            print(f"  day {d}: lag1={autocorr(sub, 1):.3f}, lag2={autocorr(sub, 2):.3f}, "
                  f"lag5={autocorr(sub, 5):.3f}")
        # per-quarter intraday
        p["q"] = pd.cut(p["timestamp"], bins=4, labels=[0, 1, 2, 3])
        for q in [0, 1, 2, 3]:
            sub = p[p["q"] == q]["wall_mid"].diff().to_numpy()
            print(f"  intraday Q{q}: lag1={autocorr(sub, 1):.3f}")
        # full lag spectrum
        lags = list(range(1, 201))
        ac = [autocorr(rets, L) for L in lags]
        ax.axhline(0, color="k", lw=0.5)
        ax.plot(lags, ac)
        ax.set_title(f"{prod}: wall_mid return autocorr")
        ax.set_xlabel("lag (ticks)"); ax.set_ylabel("rho")
    fig.tight_layout()
    fig.savefig(f"{PLOTS_DIR}/02_q4_autocorr.png", dpi=110)
    plt.show()

q4_autocorr(books)

## Q5 — Mean reversion structure

Variance ratio (VR<1 = mean-reverting), Hurst exponent on price levels and returns, and an AR(1) half-life on the demeaned price series.

In [ ]:
def variance_ratio(x: np.ndarray, k: int) -> float:
    x = x[~np.isnan(x)]
    if len(x) < k * 5:
        return np.nan
    var1 = np.var(x, ddof=1)
    rk = pd.Series(x).rolling(k).sum().dropna().to_numpy()
    vark = np.var(rk, ddof=1)
    return float(vark / (k * var1))


def hurst_rs(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    if len(x) < 200:
        return np.nan
    ns = [16, 32, 64, 128, 256, 512]
    rs_vals = []
    for n in ns:
        if n * 4 > len(x):
            break
        chunks = len(x) // n
        rs_chunk = []
        for i in range(chunks):
            seg = x[i * n:(i + 1) * n]
            mean = seg.mean()
            cum = np.cumsum(seg - mean)
            R = cum.max() - cum.min()
            S = seg.std(ddof=1)
            if S > 0:
                rs_chunk.append(R / S)
        if rs_chunk:
            rs_vals.append((np.log(n), np.log(np.mean(rs_chunk))))
    if len(rs_vals) < 3:
        return np.nan
    xs, ys = zip(*rs_vals)
    slope = np.polyfit(xs, ys, 1)[0]
    return float(slope)


def q5_meanreversion(books: dict) -> None:
    print("\n### Q5 — Mean reversion structure")
    for prod, p in books.items():
        print(f"\n[{prod}]")
        rets = p["wall_mid"].diff().to_numpy()
        for k in (2, 5, 10, 50):
            print(f"  variance ratio k={k}: {variance_ratio(rets, k):.3f} "
                  "(<1 = mean-reverting)")
        # Hurst on price levels
        h_p = hurst_rs(p["wall_mid"].dropna().to_numpy())
        # Hurst on returns (sanity)
        h_r = hurst_rs(rets[~np.isnan(rets)])
        print(f"  Hurst (price R/S): {h_p:.3f}  (Hurst returns: {h_r:.3f})")
        # AR(1) half-life on demeaned price
        s = p["wall_mid"].dropna().to_numpy()
        s_dm = s - s.mean()
        rho = np.corrcoef(s_dm[:-1], s_dm[1:])[0, 1]
        if 0 < rho < 1:
            hl = -np.log(2) / np.log(rho)
            print(f"  AR(1) on price rho={rho:.4f}, half-life={hl:.1f} ticks")
        else:
            print(f"  AR(1) on price rho={rho:.4f} (no usable half-life)")

q5_meanreversion(books)

## Q6 — Mock fill simulation (passive at wall_mid ± h)

Conservative simulator: for each h, post bid/ask at wall_mid±h and assume fill only when the next tick's best price crosses our quote. 1 unit per fill, position capped at ±200. Underestimates real fills (no aggressor model) but bounds the cross-fill rate.

In [ ]:
def simulate_mm(p: pd.DataFrame, h: int, max_pos: int = 200) -> dict:
    """Naive passive MM simulator.
    Each tick: post bid at floor(wall_mid - h), ask at ceil(wall_mid + h).
    Fill rules (per-tick, conservative):
      - If our bid >= best_ask of NEXT tick (book moved through us) -> we get filled buy at our bid.
      - If our ask <= best_bid of NEXT tick -> we get filled sell at our ask.
      - Size: assume 1 unit per fill (lower bound), capped by position limit.
    """
    wall = p["wall_mid"].to_numpy()
    ba = p["best_ask"].to_numpy()
    bb = p["best_bid"].to_numpy()
    n = len(p)
    pos = 0
    cash = 0.0
    fills_buy = 0
    fills_sell = 0
    pos_track = []
    for i in range(n - 1):
        if np.isnan(wall[i]):
            pos_track.append(pos)
            continue
        my_bid = np.floor(wall[i] - h)
        my_ask = np.ceil(wall[i] + h)
        # next-tick fill check
        nb_a = ba[i + 1]
        nb_b = bb[i + 1]
        if not np.isnan(nb_a) and my_bid >= nb_a and pos < max_pos:
            cash -= my_bid
            pos += 1
            fills_buy += 1
        if not np.isnan(nb_b) and my_ask <= nb_b and pos > -max_pos:
            cash += my_ask
            pos -= 1
            fills_sell += 1
        pos_track.append(pos)
    # mark to wall_mid at end
    final_mid = wall[~np.isnan(wall)][-1]
    pnl = cash + pos * final_mid
    return {
        "h": h, "fills_buy": fills_buy, "fills_sell": fills_sell,
        "fill_rate_per_tick": (fills_buy + fills_sell) / n,
        "avg_abs_pos": float(np.mean(np.abs(pos_track))),
        "max_abs_pos": int(np.max(np.abs(pos_track))),
        "pnl_proxy": float(pnl),
    }


def q6_fillsim(books: dict) -> None:
    print("\n### Q6 — Mock fill simulation (passive at wall_mid ± h)")
    rows = []
    for prod, p in books.items():
        for h in (2, 5, 8, 10, 15):
            r = simulate_mm(p, h)
            r["product"] = prod
            rows.append(r)
            print(f"  {prod} h={h:2d}: fill/tick={r['fill_rate_per_tick']:.3%}, "
                  f"buys={r['fills_buy']}, sells={r['fills_sell']}, "
                  f"avg|pos|={r['avg_abs_pos']:.1f}, max|pos|={r['max_abs_pos']}, "
                  f"pnl_proxy={r['pnl_proxy']:.0f}")
    df = pd.DataFrame(rows)
    df.to_csv(f"{PLOTS_DIR}/02_q6_fillsim.csv", index=False)

q6_fillsim(books)

## Q7/Q8 — Inventory drift & adverse selection

Re-simulate at h ≈ median_spread/2 and look at mid drift +5/+10/+50 ticks after each fill. Negative drift after BUY = adverse; positive net edge = we're getting paid for liquidity.

In [ ]:
def q7_q8_inventory_adverse(books: dict) -> None:
    print("\n### Q7/Q8 — Inventory drift & adverse selection")
    # We re-simulate at h=median_spread/2 and inspect mid drift after fills.
    for prod, p in books.items():
        h = int(round(p["spread"].median() / 2))
        wall = p["wall_mid"].to_numpy()
        ba = p["best_ask"].to_numpy()
        bb = p["best_bid"].to_numpy()
        n = len(p)
        pos = 0
        buy_idx, sell_idx = [], []
        for i in range(n - 1):
            if np.isnan(wall[i]):
                continue
            my_bid = np.floor(wall[i] - h)
            my_ask = np.ceil(wall[i] + h)
            if not np.isnan(ba[i + 1]) and my_bid >= ba[i + 1] and pos < 200:
                pos += 1
                buy_idx.append(i + 1)
            if not np.isnan(bb[i + 1]) and my_ask <= bb[i + 1] and pos > -200:
                pos -= 1
                sell_idx.append(i + 1)
        print(f"\n[{prod}] h={h}, buys={len(buy_idx)}, sells={len(sell_idx)}")
        for horizon in (5, 10, 50):
            buy_drift = []
            for i in buy_idx:
                if i + horizon < n:
                    buy_drift.append(wall[i + horizon] - wall[i])
            sell_drift = []
            for i in sell_idx:
                if i + horizon < n:
                    sell_drift.append(wall[i + horizon] - wall[i])
            if buy_drift and sell_drift:
                # adverse if buys see negative drift, sells see positive drift
                bd = np.mean(buy_drift); sd = np.mean(sell_drift)
                edge = bd - sd  # positive = favorable (buy then mid up, sell then mid down)
                print(f"  +{horizon:3d}t: drift after BUY ={bd:+.3f}, "
                      f"after SELL ={sd:+.3f}, edge={edge:+.3f}")

q7_q8_inventory_adverse(books)

## Summary + Key Findings

Every number from `docs/round_3/research/04_microstructure.md`. Source is `notebooks/02_microstructure_eda.py` over `data/round_3/prices_round_3_day_{0,1,2}.csv` (~30k ticks/product).

### TL;DR
- Books have only 2 visible levels per side ~98% of the time. Wall = the deeper L2 (typical size 25 hydrogel / 40 VFE), inside = thinner L1 (12 / 25). 3-level books essentially never occur.
- Wall_mid is *not* materially different from top_mid in volatility — std(d_wall)/std(d_top) ≈ 0.88 for both products. Use wall_mid as fair anchor (matches Frankfurt) but expect very similar dynamics.
- Spread is rock-stable. HYDROGEL spread = 16 on 92.7% of ticks; VFE spread = 5 on 74.2%, =6 on 18.2%. No intraday or per-day drift.
- Mean reversion is real but weak. Lag-1 autocorr -0.01 to -0.05 per day on wall_mid; variance ratios 0.90-0.98 → mild MR. AR(1) half-life ~350-380 ticks.
- Current half_edge settings defensible but `half_edge=8` on HYDROGEL is on the boundary. Cross-fill sim: passive quotes at wall_mid±8 essentially never get crossed; sit AT best bid/ask ~50% of ticks. For VFE, half_edge=2 is aggressively inside the 5-wide spread (top-of-book ~98%).

### Q1 — Depth profile
- HYDROGEL_PACK: L1-only 0%, L2 populated 98.4%, L3 populated 1.6%, both sides 3-deep 0%.
- VELVETFRUIT_EXTRACT: L1-only 45.6%, L2 populated 52.3%, L3 populated 2.0%, both sides 3-deep 0%.
- Median sizes — HYDROGEL: L1=12, L2=25, L3=25 (when present). VFE: L1=25, L2=40, L3=40.
- Exchange-MM puts the larger size at L2 → "wall mid" is meaningful.

### Q2 — Wall mid vs top mid
- mean(wall_mid - top_mid): HYDROGEL +0.012, VFE +0.019.
- std(wall_mid - top_mid): HYDROGEL 0.87, VFE 0.54.
- std(d_top_mid): HYDROGEL 2.17, VFE 1.13.
- std(d_wall_mid): HYDROGEL 1.92, VFE 0.98.
- ratio d_wall / d_top: 0.88 / 0.87.
- P(d_top = 0): HYDROGEL 18.0%, VFE 24.7%.
- P(d_wall = 0): HYDROGEL 18.8%, VFE 25.5%.
- Wall_mid is ~12% smoother than top_mid in stdev terms. Wins primarily because robust to L1-size noise (when L1 size momentarily drops, wall_mid stays anchored to the persistent deep level).

### Q3 — Spread regimes
- HYDROGEL: median 16, mean 15.72, std 1.46. 92.7% of ticks at spread=16. Anomalies are tighter (7-9, ~3% of ticks) when both walls collapse together.
- VFE: median 5, mean 4.99, std 0.85. 74.2% at spread=5, 18.2% at spread=6. Bimodal but tight.
- Per-day mean spread HYDROGEL: 15.70 / 15.73 / 15.74. Per-day mean spread VFE: 4.99 / 4.98 / 4.99. No intraday or per-day drift.

### Q4 — Autocorrelation robustness
- Per-day lag-1 wall_mid return autocorr — HYDROGEL: day 0 -0.027, day 1 -0.016, day 2 -0.011.
- Per-day lag-1 wall_mid return autocorr — VFE: day 0 -0.038, day 1 -0.045, day 2 -0.042.
- Per-quarter intraday: stable, all negative, magnitudes 0.001-0.05.
- Lags 2-200 essentially noise around 0 (lag-1 dominates spectrum).
- Earlier `01_initial_eda.md` reported -0.13/-0.16 on `mid_price`; the gap vs wall_mid suggests the prior figure was largely bid-ask bounce, not genuine fair-value MR.

### Q5 — Mean reversion structure
- Variance ratios HYDROGEL — k=2: 0.982, k=5: 0.968, k=10: 0.955, k=50: 0.901.
- Variance ratios VFE — k=2: 0.958, k=5: 0.928, k=10: 0.915, k=50: 0.928.
- Mild MR at all horizons; flattens past k=10.
- Hurst (price R/S) ≈ 1.0 (integrated process, expected). Hurst on returns ≈ 0.56 — borderline mild persistence.
- AR(1) on price level: rho≈0.998. Half-life ~380 ticks (HYDROGEL) / ~350 ticks (VFE). Too long to exploit with passive MM but consistent with slow wandering.

### Q6 — Mock fill simulation (cross-only, passive at wall_mid ± h)
- HYDROGEL h=2: cross fills 97 buy / 44 sell. Top-of-book occupancy 98% / 98%.
- HYDROGEL h=5: cross fills 3 / 1. Top-of-book 92% / 92%.
- HYDROGEL h=6: cross fills 1 / 1. Top-of-book 84% / 84%.
- HYDROGEL h=8: cross fills 0 / 0. Top-of-book **50% / 49%**.
- HYDROGEL h=10: cross fills 0 / 0. Top-of-book 15% / 15%.
- VFE h=2: cross fills 213 / 28. Top-of-book 81% / 80%.
- VFE h=3: cross fills 41 / 3. Top-of-book 47% / 48%.
- VFE h=5: cross fills 2 / 0. Top-of-book 1.9% / 1.8%.
- HYDROGEL spread=16, wall spread=21. With h=8, our quote sits ≈ AT best bid/ask (wall_mid - 8 ≈ best_bid). h=7 → inside spread always. h=10 → behind ~85% of ticks.
- VFE h=2 puts us inside the 5-wide spread, top-of-book 81%. h=3 collapses to 47%, barely any cross fills.

### Q7/Q8 — Inventory drift & adverse selection
- HYDROGEL at h=8: cannot measure (zero simulated cross fills).
- VFE at h=2 — drift after BUY / after SELL / edge:
  - +5t: +0.10 / -0.70 / **+0.79**
  - +10t: -0.06 / -0.07 / +0.01
  - +50t: -0.07 / -0.52 / +0.45
- BUY fills slightly favorable at 5t, neutral after. SELL fills happen when price was momentarily high then drops (favorable for us). Net edge positive, no severe adverse selection.
- Asymmetry: 213 buys vs 28 sells in VFE sim → predominantly accumulating long inventory. Either persistent net buying pressure on VFE or asymmetric placement of our quote vs the wall — investigate before sizing up.

### Recommendations for `src/trader.py`
1. **HYDROGEL half_edge: keep 8, but add an option to flex to 7.** At h=8 top-of-book ~50% (real fills come from passive bot hits, not crossings). h=7 → inside spread always (better fill rate, same per-trade edge=7 ticks; risk: more adverse selection). h=10 → behind 85% (don't go there). Suggest sweep h ∈ {6, 7, 8}.
2. **VFE half_edge: 2 is fine but adverse selection non-trivial.** h=2 = top-of-book 81%, real fills, edge positive. h=3 = top-of-book 47%, ~5x fewer fills. **Asymmetry warning**: 213 buys vs 28 sells at h=2 → will hit long position limit fast. Consider asymmetric quoting: `bid = wall_mid - 3`, `ask = wall_mid + 2` (skew bid wider) until the asymmetry is explained.
3. **Wall_mid is the right anchor.** ~12% lower stdev of changes vs top_mid. Keep `wall_mid()` in trader.py as-is.
4. **Both products are MM-able.** Stable wide spreads, low adverse selection, no regime shift across days. HYDROGEL is the better candidate (wider spread = more edge per round-trip, fully independent of options chain). VFE exploitable but coordinate with options team since VEV hedging hits this book.
5. **Current `skew_per_unit=0.04`** untested by this analysis. Worth a separate sweep — at +50 inventory it shifts quotes by 2 ticks: significant for VFE (40% of half_edge) but tiny for HYDROGEL (25% of half_edge).

### Per-day generalisation
- All key stats (spread, autocorr lag-1, variance ratios) stable across days 0/1/2. No regime shift.
- Per-day std of HYDROGEL price varies (25 / 38 / 32) but the *microstructure* doesn't.

### Caveats
- Fill simulator is a lower bound (cross-only). Real fills include bot aggressors hitting our resting quote → would proportionally favor inside-spread quoting (h=2 for VFE, h≤7 for HYDROGEL).
- "Top-of-book occupancy" ≠ fill rate — only means we'd be quoted at the best price; actual fill depends on bot activity not isolated by the price feed.
- Trade file (2382 rows for both products combined over 30k ticks) is sparse — ~4% of ticks have a recorded counterparty trade. Bot fill rate per tick is therefore in the low single-digit percent range, consistent with the simulated cross fill rates at small h.